# Task 8: Resume Screening Using NLP

In this notebook, we explore the datasets and test the semantic matching model (`SentenceTransformers`) used to screen resumes against job descriptions.

In [4]:
import sys
import os
import pandas as pd

# Add src to path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from preprocess import clean_text
from ner import extract_entities
from matcher import ResumeMatcher

c:\Users\eslam\OneDrive\Desktop\programming\por\Elevvo Internship\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data
We load the mock data generated in the `data/` directory.

In [5]:
data_dir = os.path.join('..', 'data')
resumes_df = pd.read_csv(os.path.join(data_dir, 'resumes.csv'))
jobs_df = pd.read_csv(os.path.join(data_dir, 'jobs.csv'))

print("Resumes shape:", resumes_df.shape)
print("Jobs shape:", jobs_df.shape)

resumes_df.head()

Resumes shape: (5, 2)
Jobs shape: (3, 3)


,id,text
0,R1,Experienced Data Scientist with 5 years in mac...
1,R2,Frontend Developer with 3 years experience. Sk...
2,R3,Recent graduate looking for a Junior Data Anal...
3,R4,"Senior Backend Engineer. Expertise in Node.js,..."
4,R5,NLP Engineer with experience in HuggingFace Tr...


## 2. Text Preprocessing and NER Extraction
Clean the text and extract technical skills using our custom rule-based and spaCy-based NER pipeline.

In [6]:
resumes_df['clean_text'] = resumes_df['text'].apply(clean_text)
jobs_df['clean_description'] = jobs_df['description'].apply(clean_text)

# Extract skills as a list
resumes_df['extracted_skills'] = resumes_df['text'].apply(lambda t: extract_entities(t)['skills'])

resumes_df[['id', 'extracted_skills']]

,id,extracted_skills
0,R1,"[Python, Scikit-learn, Machine learning, Nlp, ..."
1,R2,"[Html, Java, React, Css, Javascript]"
2,R3,"[Sql, Python]"
3,R4,"[Python, Postgresql, Node.js, Sql, Aws]"
4,R5,"[Pytorch, Spacy, Nlp]"


## 3. Resume Matching via Sentence Transformers
We use `all-MiniLM-L6-v2` to compute the cosine similarity between the job description and candidate resumes.

In [ ]:
matcher = ResumeMatcher()

target_job = jobs_df.iloc[0]
print(f"Target Job: {target_job['title']}\n")

ranked_resumes = matcher.rank_resumes(
    job_desc=target_job['clean_description'],
    resumes=resumes_df['clean_text'].tolist(),
    resume_ids=resumes_df['id'].tolist()
)

pd.DataFrame(ranked_resumes)

Loading SentenceTransformer model 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14788.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Target Job: Machine Learning Engineer



,id,score,text
0,R1,0.715595,experienced data scientist with 5 years in mac...
1,R5,0.481213,nlp engineer with experience in huggingface tr...
2,R3,0.470028,recent graduate looking for a junior data anal...
3,R4,0.392116,"senior backend engineer. expertise in node.js,..."
4,R2,0.378369,frontend developer with 3 years experience. sk...
